<a href="https://colab.research.google.com/github/Shams-Sajid-Rahman/Sajid_FlyRank_AI/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: Refresh / Content Opportunity Scoring (Lane 2).**

I'm picking this lane because it directly extends the two starter notebooks I already ran: notebook 01 surfaced a real CTR pattern by content type and position tier, and notebook 02 showed a hand-written rule and a readable decision tree competing on the same task — ranking pages for review. Lane 2 asks a sharper version of that same question: *which pages should a content reviewer look at first, out of thousands, given limited time?* That is a decision someone at FlyRank (or a client's content team) actually makes every week, it has a clear action (refresh / expand / protect / monitor), and the starter pipeline already proves a learned ranking beats a fixed rule on this exact data (see numbers below). I also have full-stack + ML background from my capstone (MedT2V) and IEEE paper, both of which involved building and evaluating models against a baseline, so this lane lets me reuse that instinct: baseline first, then earn any extra complexity.

In [ ]:
# nothing to compute for this section

## 2. The question: decision, action, cost of a wrong call

**Research question:** Among a client's content pages, which pages should a content reviewer prioritize first for refresh, given limited review capacity?

- **Unit of analysis:** one page (`content_id`), aggregated over a trailing 90-day window.
- **Decision this improves:** which pages a content reviewer opens first when they only have time to review a handful this week/sprint — not "will this page ever recover."
- **Who acts, and how:** a content reviewer or SEO strategist at FlyRank (or a client's in-house team) works down a ranked queue and either refreshes the page, expands it, protects it, or leaves it on monitor — the same action set the starter pipeline already outputs.
- **Output:** a ranked list of pages with a score and a reason code (why this page was flagged), not a single number in isolation.
- **Cost of a wrong call:**
  - *False positive* (flagged as worth reviewing, but it wasn't): wastes a reviewer's limited hours on a page that didn't need attention.
  - *False negative* (a genuinely declining/opportunity page never surfaces): the page keeps losing visibility silently until someone notices by accident, which is the expensive failure mode since traffic keeps leaking the whole time.
  - Because reviewer time is the scarce resource, precision in the top of the queue (Precision@20 / Precision@50) matters more than overall accuracy — a queue that's 90% right on average but wrong in its top 10 is worse than one that's right where a human will actually look.
- **Why data/ML helps at all:** a single hand-written rule (stale + visible) can be sharp but narrow — in the starter data it only catches 17 of 30,000 pages, even though those 17 are overwhelmingly declining. That means most real decline patterns live outside that one rule, in a mix of position, freshness, word count, and traffic signals that's too tangled to write by hand but small enough (2-3 features deep) that a simple, readable model can still find and explain it — which the starter pipeline already demonstrates (numbers below).

In [ ]:
# nothing to compute for this section

## 3. Quick look at the data (2-3 real numbers)

Loading the starter dataset and pulling numbers that support Lane 2, using only observable signals (no product decision flags).

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Shams-Sajid-Rahman/Sajid_FlyRank_AI"
REPO_DIR = "Sajid_FlyRank_AI"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working directory:", os.getcwd())

Working directory: /content/Sajid_FlyRank_AI


In [2]:
import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

n_rows = len(df)
n_clients = df["client_id"].nunique()
decline_rate = df["is_declining_label"].mean()

stale = df["days_since_last_update"] >= 180
visible = df["impressions_90d"] >= 500
stale_visible = stale & visible

decline_rate_in_rule = df.loc[stale_visible, "is_declining_label"].mean()
decline_rate_outside_rule = df.loc[~stale_visible, "is_declining_label"].mean()

print(f"Pages: {n_rows:,}  |  Clients: {n_clients}")
print(f"Overall declining rate: {decline_rate:.1%}")
print(f"'stale + visible' pages: {stale_visible.sum()} of {n_rows:,} ({stale_visible.mean():.2%} of all pages)")
print(f"Declining rate INSIDE that rule: {decline_rate_in_rule:.1%}   (n={stale_visible.sum()})")
print(f"Declining rate OUTSIDE that rule: {decline_rate_outside_rule:.1%}")

Pages: 30,000  |  Clients: 32
Overall declining rate: 54.2%
'stale + visible' pages: 17 of 30,000 (0.06% of all pages)
Declining rate INSIDE that rule: 94.1%   (n=17)
Declining rate OUTSIDE that rule: 54.2%


**Reading these numbers:** the "stale + visible" hand rule is sharp (94% of the pages it catches are declining) but tiny — it only flags 17 out of 30,000 pages (0.06%), so on its own it leaves almost the entire inventory unscored. That's the exact gap Lane 2 exists to close.

A second, independently useful number: the starter pipeline (`scripts/03_train_model.py`, results committed in `outputs/model_report.md`) already ran a proper client-holdout comparison on this same data:

In [3]:
# From outputs/model_report.md — already run with client-holdout validation (whole clients held out of training)
comparison = pd.DataFrame({
    "method": ["baseline_rules", "logistic_regression", "decision_tree", "random_forest"],
    "precision_at_50": [0.240, 0.400, 0.540, 0.740],
})
comparison["pages_right_of_top_50"] = (comparison["precision_at_50"] * 50).astype(int)
comparison

,method,precision_at_50,pages_right_of_top_50
0,baseline_rules,0.24,12
1,logistic_regression,0.40,20
2,decision_tree,0.54,27
3,random_forest,0.74,37


The rule-based baseline gets 12 of its top 50 recommendations right; a random forest gets 37 of 50 right — on the *same held-out clients*, not the same clients it trained on. That's a 3x lift in reviewer hit-rate at the exact point (top of the queue) where reviewer time is spent, which is the strongest single number backing this lane.

## 4. Careful words: what I can and can't claim


**What this work CAN claim:**
- Observed associations between page signals (staleness, visibility, position, word count, CTR) and a *current-window* decline label.
- A directional ranking: "pages with these characteristics are more likely to be in the declining bucket than pages without them," on this starter slice, with client-holdout validation.
- Decision-support value: a ranked queue that puts a reviewer's limited hours on higher-probability pages first, beating the plain rule baseline on Precision@K.

**What this work CANNOT claim:**
- That refreshing a flagged page will *cause* it to recover — that needs an experiment (e.g. before/after with a control group), not this observational data.
- That any result "proves" a Google ranking algorithm factor — I only observe FlyRank's own measured signals (impressions, CTR, position), never Google's internals.
- That the starter label is the ideal target — `is_declining_label` is a proxy derived from the *current* 90-day window's `trend_direction`, not a *future* outcome. A stronger capstone version of this lane should move to a forward-looking label (prior 90 days of features → decline over the *next* 30 days) to avoid the proxy-label weakness the lane guide calls out.
- That results on this 30,000-row, 32-client anonymized starter slice generalize to the full ~79M-row warehouse without being re-earned there.

I'll use "observed," "measured," "directional," and "decision-support" — never "proves" or "causes."

In [ ]:
# (no code needed for this section — it is a written commitment, not a computation)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [4]:
#all done by Sajid